In [ ]:
# ==============================================================================
# AUDITORIA SINAN: RASTREAMENTO AMPLIADO COM GRÁFICO 100% EMPILHADO (MINIMALISTA)
# ==============================================================================

import pandas as pd
import numpy as np
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 1. CARREGAMENTO DOS DADOS
# ------------------------------------------------------------------------------
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except FileNotFoundError:
    print(f"Erro: O arquivo {file_path} não foi encontrado no diretório atual.")
    raise

cols_sinan = [c for c in df.columns if 'sinan' in c.lower() or 'notifica' in c.lower()]
if not cols_sinan:
    raise ValueError("Coluna do SINAN não encontrada.")
col_sinan = cols_sinan[0]

# ------------------------------------------------------------------------------
# 2. MOTOR DE BUSCA CLÍNICA (AMPLIADO)
# ------------------------------------------------------------------------------
def categorizar_doenca_ampliada(nome_coluna):
    col_lower = nome_coluna.lower()
    termos_proibidos = ['exame', 'teste', 'sorologia', 'rastreio', 'solicita', 'vacina', 'suspeita', 'pedido']
    if any(t in col_lower for t in termos_proibidos):
        return None

    if any(k in col_lower for k in ['hiv', 'aids', 'sida']): return 'HIV/AIDS'
    if any(k in col_lower for k in ['sifilis', 'sífilis']): return 'Sífilis'
    if 'tuberculose' in col_lower: return 'Tuberculose'
    if 'hepatite' in col_lower: return 'Hepatite'
    if 'dengue' in col_lower: return 'Dengue'
    if 'hanseníase' in col_lower: return 'Hanseníase'
    if 'covid' in col_lower: return 'COVID-19'
    if 'meningite' in col_lower: return 'Meningite'
    if 'malária' in col_lower: return 'Malária'
    if any(k in col_lower for k in ['varíola', 'mpox', 'monkeypox']): return 'Mpox/Varíola'
    if any(k in col_lower for k in ['zika', 'chikungunya']): return 'Arboviroses (Zika/Chik)'
    if 'leishmaniose' in col_lower: return 'Leishmaniose'
    if 'chagas' in col_lower: return 'Doença de Chagas'
    if 'leptospirose' in col_lower: return 'Leptospirose'
    if 'esquistossomose' in col_lower: return 'Esquistossomose'
    if 'toxoplasmose' in col_lower: return 'Toxoplasmose'
    if 'raiva' in col_lower or 'mordedura' in col_lower or 'animal' in col_lower: return 'Atendimento Antirrábico'
    if 'violência' in col_lower or 'violencia' in col_lower or 'suicídio' in col_lower: return 'Violência / Autoprovocada'
    if 'intoxicação' in col_lower or 'intoxicacao' in col_lower or 'veneno' in col_lower: return 'Intoxicação Exógena'
    if 'peçonhento' in col_lower or 'escorpião' in col_lower or 'cobra' in col_lower: return 'Acidente c/ Peçonhento'

    return None

mapa_colunas = {}
for c in df.columns:
    categoria = categorizar_doenca_ampliada(c)
    if categoria:
        mapa_colunas[c] = categoria

doencas_detectadas = sorted(list(set(mapa_colunas.values())))

# ------------------------------------------------------------------------------
# 3. LÓGICA DE AUDITORIA E CRUZAMENTO
# ------------------------------------------------------------------------------
valores_vazios = ['nan', '', 'não', 'nao', 'none', 'false', '0']
df['Tem_SINAN'] = ~df[col_sinan].astype(str).str.strip().str.lower().isin(valores_vazios)
valores_positivos = ['checked', 'sim', 'yes', 'verdadeiro', 'true', '1']

for doenca in doencas_detectadas:
    cols_desta_doenca = [c for c, d in mapa_colunas.items() if d == doenca]
    df[doenca] = df[cols_desta_doenca].apply(lambda row: row.astype(str).str.strip().str.lower().isin(valores_positivos).any(), axis=1)

agg_dict = {doenca: 'max' for doenca in doencas_detectadas}
agg_dict['Tem_SINAN'] = 'max'

df_pacientes = df.groupby('Record ID').agg(agg_dict).reset_index()

df_melted = df_pacientes.melt(id_vars=['Record ID', 'Tem_SINAN'],
                              value_vars=doencas_detectadas,
                              var_name='Doenca',
                              value_name='Tem_Doenca')

df_auditoria = df_melted[df_melted['Tem_Doenca'] == True].copy()
df_auditoria['Status_SINAN'] = np.where(df_auditoria['Tem_SINAN'], 'Sim (Notificado na Base)', 'Não (Falta Notificar)')

# ------------------------------------------------------------------------------
# 4. GERAÇÃO DO GRÁFICO (ESTÉTICA EXATA DA IMAGEM)
# ------------------------------------------------------------------------------
if df_auditoria.empty:
    print("Nenhuma doença compulsória confirmada foi encontrada nesta base.")
else:
    # Prepara os dados matemáticos
    df_agrupado = df_auditoria.groupby(['Doenca', 'Status_SINAN']).size().reset_index(name='Casos')
    df_totais = df_auditoria.groupby('Doenca').size().reset_index(name='Total')
    df_agrupado = pd.merge(df_agrupado, df_totais, on='Doenca')
    df_agrupado['Percentual'] = (df_agrupado['Casos'] / df_agrupado['Total']) * 100
    
    # Ordena as doenças pelo total de casos (maiores no topo)
    df_agrupado = df_agrupado.sort_values(by='Total', ascending=True)

    # Força a ordem das cores: Vermelho (Não) primeiro, Verde (Sim) depois
    df_agrupado['Status_SINAN'] = pd.Categorical(
        df_agrupado['Status_SINAN'], 
        categories=['Não (Falta Notificar)', 'Sim (Notificado na Base)'], 
        ordered=True
    )
    df_agrupado = df_agrupado.sort_values(['Doenca', 'Status_SINAN'])

    # Criação do Gráfico
    fig = px.bar(
        df_agrupado,
        y='Doenca',
        x='Percentual',
        color='Status_SINAN',
        orientation='h',
        text=df_agrupado.apply(lambda row: f"{row['Casos']} casos ({row['Percentual']:.1f}%)", axis=1),
        color_discrete_map={
            'Não (Falta Notificar)': '#d9534f',      # Vermelho exato da imagem
            'Sim (Notificado na Base)': '#5cb85c'    # Verde exato da imagem
        }
    )

    # Layout Minimalista (Zero eixos, apenas rótulos)
    fig.update_layout(
        barmode='stack',
        height=200 + (len(df_totais) * 80), # Altura dinâmica para as barras não ficarem espremidas
        title=dict(
            text=f"Auditoria SINAN (Diagnósticos e Encaminhamentos) - Total: {len(df_auditoria)} Casos",
            font=dict(size=22, color="#333", family="Arial"),
            x=0.01, # Alinhado à esquerda
            y=0.95
        ),
        xaxis=dict(visible=False), # Esconde o eixo X completamente (linhas, números, títulos)
        yaxis=dict(
            title='', 
            showline=False, 
            showgrid=False, 
            zeroline=False, 
            tickfont=dict(size=14, color="#333")
        ),
        legend_title_text='',
        legend=dict(
            orientation="h", 
            yanchor="bottom", 
            y=1.02, 
            xanchor="center", 
            x=0.5, 
            font=dict(size=14),
            itemclick=False, 
            itemdoubleclick=False
        ),
        margin=dict(l=10, r=20, t=120, b=20),
        plot_bgcolor='white',
        paper_bgcolor='white',
        bargap=0.3 # Espaço elegante entre as barras horizontais
    )

    # O segredo do alinhamento do texto: insidetextanchor='end'
    fig.update_traces(
        textposition='inside',
        insidetextanchor='end', # Empurra o texto para a extremidade direita de cada cor
        textfont=dict(color='white', size=14, family="Arial Black"),
        marker_line_width=0
    )
    
    fig.show()

    # ------------------------------------------------------------------------------
    # 5. EXPORTAÇÃO DA TABELA
    # ------------------------------------------------------------------------------
    df_tabela = df_agrupado.copy()
    df_tabela['Percentual'] = df_tabela['Percentual'].round(1)
    df_tabela = df_tabela.sort_values(by=['Total', 'Percentual'], ascending=[False, False]).reset_index(drop=True)

    df_tabela = df_tabela.rename(columns={
        'Doenca': 'Agravo / Doença',
        'Status_SINAN': 'Status de Notificação',
        'Casos': 'Quantidade de Pacientes',
        'Total': 'Total de Casos do Agravo',
        'Percentual': 'Proporção (%)'
    })

    nome_arquivo = 'Tabela_Auditoria_SINAN_Ampliada.csv'
    df_tabela.to_csv(nome_arquivo, index=False, encoding='utf-8-sig')

    print(f"\n✅ Gráfico gerado perfeitamente! Tabela salva como: '{nome_arquivo}'.\n")

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')

# 1. CARREGAMENTO DOS DADOS
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except FileNotFoundError:
    print(f"Erro: O arquivo {file_path} não foi encontrado.")
    raise

# 2. MAPEAMENTO DE COLUNAS
def categorizar_doenca_ampliada(nome_coluna):
    col_lower = nome_coluna.lower()
    termos_proibidos = ['exame', 'teste', 'sorologia', 'rastreio', 'solicita', 'vacina', 'suspeita', 'pedido']
    if any(t in col_lower for t in termos_proibidos):
        return None

    if any(k in col_lower for k in ['hiv', 'aids', 'sida']): return 'HIV/AIDS'
    if any(k in col_lower for k in ['sifilis', 'sífilis']): return 'Sífilis'
    if 'tuberculose' in col_lower: return 'Tuberculose'
    if 'hepatite' in col_lower: return 'Hepatite'
    if 'dengue' in col_lower: return 'Dengue'
    if 'hanseníase' in col_lower: return 'Hanseníase'
    if 'covid' in col_lower: return 'COVID-19'
    if 'meningite' in col_lower: return 'Meningite'
    if 'malária' in col_lower: return 'Malária'
    if any(k in col_lower for k in ['varíola', 'mpox', 'monkeypox']): return 'Mpox/Varíola'
    if any(k in col_lower for k in ['zika', 'chikungunya']): return 'Arboviroses (Zika/Chik)'
    if 'leishmaniose' in col_lower: return 'Leishmaniose'
    if 'chagas' in col_lower: return 'Doença de Chagas'
    if 'leptospirose' in col_lower: return 'Leptospirose'
    if 'esquistossomose' in col_lower: return 'Esquistossomose'
    if 'toxoplasmose' in col_lower: return 'Toxoplasmose'
    return None

mapa_colunas_doencas = {}
for c in df.columns:
    categoria = categorizar_doenca_ampliada(c)
    if categoria:
        mapa_colunas_doencas[c] = categoria

doencas_detectadas = list(set(mapa_colunas_doencas.values()))
cols_oftalmo = [c for c in df.columns if 'oftalmo' in c.lower()]

# 3. CRUZAMENTO DE DADOS (POR PACIENTE ÚNICO)
valores_positivos = ['checked', 'sim', 'yes', 'verdadeiro', 'true', '1']

df['Tem_Oftalmo'] = df[cols_oftalmo].apply(lambda row: row.astype(str).str.strip().str.lower().isin(valores_positivos).any(), axis=1)

for doenca in doencas_detectadas:
    cols_desta_doenca = [c for c, d in mapa_colunas_doencas.items() if d == doenca]
    df[doenca] = df[cols_desta_doenca].apply(lambda row: row.astype(str).str.strip().str.lower().isin(valores_positivos).any(), axis=1)

agg_dict = {doenca: 'max' for doenca in doencas_detectadas}
agg_dict['Tem_Oftalmo'] = 'max'

df_pacientes = df.groupby('Record ID').agg(agg_dict).reset_index()

df_melted = df_pacientes.melt(id_vars=['Record ID', 'Tem_Oftalmo'],
                              value_vars=doencas_detectadas,
                              var_name='Doenca',
                              value_name='Tem_Doenca')

df_analise = df_melted[df_melted['Tem_Doenca'] == True].copy()
df_analise['Status_Encaminhamento'] = np.where(df_analise['Tem_Oftalmo'], 'Oftalmologia Solicitada', 'Sem Avaliação Ocular')

# 4. GERAÇÃO DO GRÁFICO (RELAÇÃO DOENÇA INFECCIOSA -> OFTALMO)
if df_analise.empty:
    print("Nenhuma doença compulsória confirmada foi encontrada nesta base.")
else:
    df_agrupado = df_analise.groupby(['Doenca', 'Status_Encaminhamento']).size().reset_index(name='Casos')
    df_totais = df_analise.groupby('Doenca').size().reset_index(name='Total')
    df_agrupado = pd.merge(df_agrupado, df_totais, on='Doenca')
    df_agrupado['Percentual'] = (df_agrupado['Casos'] / df_agrupado['Total']) * 100
    df_agrupado = df_agrupado.sort_values(by=['Total', 'Percentual'], ascending=[True, False])

    fig = px.bar(
        df_agrupado,
        y='Doenca',
        x='Percentual',
        color='Status_Encaminhamento',
        orientation='h',
        text=df_agrupado.apply(lambda row: f"{row['Casos']} casos ({row['Percentual']:.1f}%)", axis=1),
        color_discrete_map={
            'Oftalmologia Solicitada': '#27ae60', # Verde sólido
            'Sem Avaliação Ocular': '#bdc3c7'      # Cinza (Sem encaminhamento)
        }
    )

    # AJUSTES DE TAMANHO PARA FICAR COMPACTO
    num_doencas = len(df_totais)
    altura_dinamica = 150 + (num_doencas * 50) # Altura menor e auto-ajustável

    fig.update_layout(
        height=altura_dinamica,
        font=dict(size=13, family="Arial", color="#333"),
        xaxis=dict(range=[0, 100], showgrid=False, showticklabels=False, zeroline=False, title=''),
        yaxis=dict(showgrid=False, zeroline=False, title=''),
        legend_title_text='',
        legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
        margin=dict(l=150, r=20, t=50, b=20), # Margem superior (t) menor
        plot_bgcolor='white',
        paper_bgcolor='white',
        barmode='stack'
    )

    fig.update_traces(
        textposition='inside',
        textfont=dict(color='white', size=12, family="Arial Black"),
        marker_line_width=0
    )
    fig.show()

    # ==============================================================================
    # 5. EXPORTAÇÃO SILENCIOSA DA TABELA
    # ==============================================================================
    df_tabela = df_agrupado.copy()
    df_tabela['Percentual'] = df_tabela['Percentual'].round(1)

    # Ordena com o maior volume de casos no topo para facilitar a leitura no Excel
    df_tabela = df_tabela.sort_values(by=['Total', 'Percentual'], ascending=[False, False]).reset_index(drop=True)

    # Renomeia colunas para ficar com cara de relatório final
    df_tabela = df_tabela.rename(columns={
        'Doenca': 'Doença Infecciosa',
        'Status_Encaminhamento': 'Status de Avaliação Ocular',
        'Casos': 'Quantidade de Pacientes',
        'Total': 'Total de Casos da Doença',
        'Percentual': 'Proporção (%)'
    })

    nome_arquivo = 'Tabela_Oftalmo_Infecciosas.csv'
    df_tabela.to_csv(nome_arquivo, index=False, encoding='utf-8-sig')

    print(f"\n✅ Sucesso! Gráfico gerado rastreando {len(df_analise)} casos confirmados.")
    print(f"📂 Tabela base salva silenciosamente em: '{nome_arquivo}'.\n")

In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CARREGAMENTO DOS DADOS
# ==============================================================================
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except FileNotFoundError:
    print(f"Erro: O arquivo {file_path} não foi encontrado no diretório atual.")
    raise

# Localiza a coluna do SINAN
cols_sinan = [c for c in df.columns if 'sinan' in c.lower() or 'notifica' in c.lower()]
if not cols_sinan:
    raise ValueError("Coluna do SINAN não encontrada.")
col_sinan = cols_sinan[0]

# ==============================================================================
# 2. MOTOR DE BUSCA (DOENÇAS DE NOTIFICAÇÃO COMPULSÓRIA)
# ==============================================================================
def categorizar_doenca_ampliada(nome_coluna):
    col_lower = nome_coluna.lower()
    termos_proibidos = ['exame', 'teste', 'sorologia', 'rastreio', 'solicita', 'vacina', 'suspeita', 'pedido']
    if any(t in col_lower for t in termos_proibidos):
        return None

    if any(k in col_lower for k in ['hiv', 'aids', 'sida']): return 'HIV/AIDS'
    if any(k in col_lower for k in ['sifilis', 'sífilis']): return 'Sífilis'
    if 'tuberculose' in col_lower: return 'Tuberculose'
    if 'hepatite' in col_lower: return 'Hepatite'
    if 'dengue' in col_lower: return 'Dengue'
    if 'hanseníase' in col_lower: return 'Hanseníase'
    if 'covid' in col_lower: return 'COVID-19'
    if 'meningite' in col_lower: return 'Meningite'
    if 'malária' in col_lower: return 'Malária'
    if any(k in col_lower for k in ['varíola', 'mpox', 'monkeypox']): return 'Mpox/Varíola'
    if any(k in col_lower for k in ['zika', 'chikungunya']): return 'Arboviroses (Zika/Chik)'
    if 'leishmaniose' in col_lower: return 'Leishmaniose'
    if 'chagas' in col_lower: return 'Doença de Chagas'
    if 'leptospirose' in col_lower: return 'Leptospirose'
    if 'esquistossomose' in col_lower: return 'Esquistossomose'
    if 'toxoplasmose' in col_lower: return 'Toxoplasmose'
    if 'raiva' in col_lower or 'mordedura' in col_lower or 'animal' in col_lower: return 'Atendimento Antirrábico'
    if 'violência' in col_lower or 'violencia' in col_lower or 'suicídio' in col_lower: return 'Violência / Autoprovocada'
    if 'intoxicação' in col_lower or 'intoxicacao' in col_lower or 'veneno' in col_lower: return 'Intoxicação Exógena'
    if 'peçonhento' in col_lower or 'escorpião' in col_lower or 'cobra' in col_lower: return 'Acidente c/ Peçonhento'

    return None

mapa_colunas = {}
for c in df.columns:
    categoria = categorizar_doenca_ampliada(c)
    if categoria:
        mapa_colunas[c] = categoria

doencas_detectadas = sorted(list(set(mapa_colunas.values())))

# ==============================================================================
# 3. AUDITORIA DE PREENCHIMENTO (PACIENTE ÚNICO)
# ==============================================================================
valores_positivos = ['checked', 'sim', 'yes', 'verdadeiro', 'true', '1']

# Verifica quais pacientes têm pelo menos uma doença compulsória
for doenca in doencas_detectadas:
    cols_desta_doenca = [c for c, d in mapa_colunas.items() if d == doenca]
    df[doenca] = df[cols_desta_doenca].apply(lambda row: row.astype(str).str.strip().str.lower().isin(valores_positivos).any(), axis=1)

# Verifica se o SINAN foi preenchido (Ignora vazios, 'não', zeros, etc)
valores_vazios = ['nan', '', 'não', 'nao', 'none', 'false', '0']
df['Tem_SINAN'] = ~df[col_sinan].astype(str).str.strip().str.lower().isin(valores_vazios)

# Agrupa por paciente (Se teve a doença alguma vez na vida, e se anotou o SINAN alguma vez)
agg_dict = {doenca: 'max' for doenca in doencas_detectadas}
agg_dict['Tem_SINAN'] = 'max'

df_pacientes = df.groupby('Record ID').agg(agg_dict).reset_index()

# Derrete a base para contar os casos
df_melted = df_pacientes.melt(id_vars=['Record ID', 'Tem_SINAN'],
                              value_vars=doencas_detectadas,
                              var_name='Doenca',
                              value_name='Tem_Doenca')

# Isola apenas os casos CONFIRMADOS de doenças compulsórias
df_auditoria = df_melted[df_melted['Tem_Doenca'] == True].copy()

# ==============================================================================
# 4. CÁLCULO DAS MÉTRICAS
# ==============================================================================
total_casos = len(df_auditoria)
casos_com_sinan = df_auditoria['Tem_SINAN'].sum()
casos_em_branco = total_casos - casos_com_sinan

if total_casos > 0:
    taxa_efetividade = (casos_com_sinan / total_casos) * 100
else:
    taxa_efetividade = 0.0

# ==============================================================================
# 5. EXIBIÇÃO DO PAINEL DE GESTÃO (FORMATO EXATO DA IMAGEM)
# ==============================================================================
print("="*80)
print("SÍNTESE ESTATÍSTICA: VIGILÂNCIA EPIDEMIOLÓGICA (SINAN)")
print("="*80)
print(f"Total de casos diagnosticados com notificação compulsória: {total_casos}")
print(f"-> Casos com número do SINAN gerado/anotado: {casos_com_sinan}")
print(f"-> Casos com número do SINAN em branco:      {casos_em_branco}")
print("\n" + f"Taxa de Efetividade de Registro: {taxa_efetividade:.2f}%")
print("="*80)

if taxa_efetividade < 80: # Se for menor que 80%, dispara o alerta
    print("⚠️ ALERTA DE GESTÃO: A taxa de preenchimento está abaixo do ideal.")
    print("Recomenda-se implementar travas no sistema (campos obrigatórios) para impedir o")
    print("fechamento do prontuário sem a inclusão do número da ficha do SINAN.")